In [15]:
import pandas as pd
import numpy as np
import os

# --- 1. CONFIGURAZIONE ---
reports_dir = os.path.join("reports", "metrics")
file_path = os.path.join("dataset", "all_pr_type.csv")
ROBUST_THRESHOLD = 10
MIN_ABSOLUTE = 1

if not os.path.exists(reports_dir): os.makedirs(reports_dir)

# --- 2. CARICAMENTO ---
df_main = pd.read_csv(file_path)
df_main['created_at'] = pd.to_datetime(df_main['created_at'])
df_main['merged_at'] = pd.to_datetime(df_main['merged_at'])
df_main['time_hrs'] = (df_main['merged_at'] - df_main['created_at']).dt.total_seconds() / 3600
df_main = df_main[(df_main['merged_at'].isna()) | (df_main['time_hrs'] > 0.001)].copy()
df_main['is_merged'] = df_main['merged_at'].notna()

df_ai = df_main[df_main['agent'] != 'Human'].copy()
ai_ids = df_ai['id'].unique().tolist()

# --- 3. METADATI (PARQUET) ---
df_comm = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_comments.parquet", columns=['pr_id'], filters=[('pr_id', 'in', ai_ids)])
comm_count = df_comm.groupby('pr_id').size().reset_index(name='n_comments')

df_det = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet",
                         columns=['pr_id', 'additions', 'deletions', 'commit_stats_total', 'filename'],
                         filters=[('pr_id', 'in', ai_ids)])

df_det['directory'] = df_det['filename'].apply(lambda x: os.path.dirname(str(x)) if pd.notnull(x) else "")
pr_stats = df_det.groupby('pr_id').agg({
    'additions': 'sum', 'deletions': 'sum', 'commit_stats_total': 'sum', 'filename': 'nunique'
}).reset_index()

dir_stats = df_det[df_det['filename'].notna()].groupby('pr_id')['directory'].nunique().reset_index(name='dir_count')
pr_stats = pr_stats.merge(dir_stats, on='pr_id', how='left').fillna(0)
pr_stats['ASI_pr'] = np.where(pr_stats['filename'] > 0, (pr_stats['dir_count'] / pr_stats['filename']).clip(upper=1.0), 0)
pr_stats['PCD_pr'] = np.where(pr_stats['filename'] > 0, pr_stats['commit_stats_total'] / pr_stats['filename'], 0)

# FIX KEYERROR: Rimuoviamo additions/deletions dal df_ai prima del merge perché useremo quelle più precise del parquet
df_ai_clean = df_ai.drop(columns=['additions', 'deletions'], errors='ignore')
df_merged = df_ai_clean.merge(comm_count, left_on='id', right_on='pr_id', how='left') \
                       .merge(pr_stats, left_on='id', right_on='pr_id', how='left').fillna(0)

# --- 4. LOGICA SCR ---
df_base_scr = df_ai[['id', 'type', 'agent', 'repo_id', 'merged_at']].dropna(subset=['merged_at']).copy()
df_fix_scr = df_ai[df_ai['type'] == 'fix'][['id', 'agent', 'repo_id', 'created_at']]
df_scr_check = df_base_scr.merge(df_fix_scr, on=['agent', 'repo_id'], suffixes=('_task', '_fix'))
df_scr_check['delta'] = (df_scr_check['created_at'] - df_scr_check['merged_at']).dt.total_seconds() / 3600


# --- 4. LOGICA SCR PRECISA (CON OVERLAP DIRECTORY) ---
print("Calcolo SCR granulare con verifica Overlap Directory...")

# Prepariamo i set di directory toccate da ogni PR
pr_dir_map = df_det.groupby('pr_id')['directory'].apply(set).to_dict()

# Prepariamo i dataset base per il confronto
df_base_scr = df_ai[['id', 'type', 'agent', 'repo_id', 'merged_at']].dropna(subset=['merged_at']).copy()
df_fix_scr = df_ai[df_ai['type'] == 'fix'][['id', 'agent', 'repo_id', 'created_at']]

# Join per trovare potenziali correzioni
df_scr_check = df_base_scr.merge(df_fix_scr, on=['agent', 'repo_id'], suffixes=('_task', '_fix'))

# Calcolo Delta Temporale
df_scr_check['delta'] = (df_scr_check['created_at'] - df_scr_check['merged_at']).dt.total_seconds() / 3600

# Verifica dell'Overlap (Stessa cartella)
def check_overlap(row):
    # Recuperiamo i set di cartelle per il task e per il fix
    dirs_task = pr_dir_map.get(row['id_task'], set())
    dirs_fix = pr_dir_map.get(row['id_fix'], set())
    # Se hanno almeno una cartella in comune, non sono disgiunti
    return not dirs_task.isdisjoint(dirs_fix)

# Applichiamo il filtro dell'overlap (solo se ci sono match temporali, per performance)
mask_48h = (df_scr_check['delta'] > 0) & (df_scr_check['delta'] <= 48)
df_scr_check.loc[mask_48h, 'has_overlap'] = df_scr_check[mask_48h].apply(check_overlap, axis=1)

# Definiamo le regressioni reali: Entro 48h E nello stesso percorso
regressions = df_scr_check[(mask_48h) & (df_scr_check['has_overlap'] == True)]

def get_precise_scr(group_col):
    if regressions.empty:
        base = df_base_scr[group_col].drop_duplicates() if isinstance(group_col, str) else df_base_scr[group_col].drop_duplicates()
        if isinstance(base, pd.Series): base = base.to_frame()
        base['SCR'] = 0.0
        return base

    tasks_with_regression = regressions['id_task'].unique()
    df_base_scr['is_failed'] = df_base_scr['id'].isin(tasks_with_regression)
    return df_base_scr.groupby(group_col)['is_failed'].mean().reset_index(name='SCR')

scr_agent = get_precise_scr('agent')
scr_type = get_precise_scr('type')
scr_agent_type = get_precise_scr(['agent', 'type'])

# --- 5. FUNZIONE METRICHE ---
def compute_final_metrics(df_grouped, grouping_col, scr_df):
    # Clipping tempi per media
    upper_limit = df_grouped['time_hrs'].quantile(0.99)
    df_grouped['time_hrs_clipped'] = df_grouped['time_hrs'].clip(upper=upper_limit)

    rep = df_grouped.groupby(grouping_col).agg({
        'is_merged': 'mean', 'time_hrs': 'median', 'time_hrs_clipped': 'mean',
        'n_comments': 'mean', 'filename': ['median', 'mean'],
        'additions': 'sum', 'deletions': 'sum', # Ora queste colonne esistono univocamente
        'ASI_pr': 'mean', 'PCD_pr': 'mean', 'id': 'count'
    }).reset_index()

    cols = [grouping_col] if isinstance(grouping_col, str) else grouping_col
    rep.columns = cols + ['acc_rate', 't_med', 't_mean', 'c_avg', 'f_med', 'f_avg', 'add_sum', 'del_sum', 'ASI', 'PCD', 'sample']

    # Indici
    rep['i_struct'] = (rep['f_avg'] / (rep['f_med'] + 0.001))
    rep['i_proc'] = (rep['t_mean'] / (rep['t_med'] + 0.001))
    rep['GRS'] = 1 / np.log1p(rep['i_struct'] + rep['i_proc'] + 1)

    for c in ['i_struct', 'i_proc']:
        lim = rep[c].quantile(0.95)
        temp = rep[c].clip(upper=lim)
        rep[f'{c}_n'] = (temp - temp.min()) / (temp.max() - temp.min() + 0.001)

    rep['CDI'] = (rep['i_struct_n'] + rep['i_proc_n']) / 2
    rep['ACE'] = (rep['del_sum'] / (rep['add_sum'] + 1))
    rep['SFI'] = (rep['c_avg'] / (rep['f_avg'] + 0.001))
    rep['ASR'] = (rep['acc_rate'] / (rep['t_med'] + 0.001))

    # Join SCR
    if isinstance(grouping_col, list):
        rep = rep.merge(scr_agent_type, on=grouping_col, how='left')
    else:
        rep = rep.merge(scr_df, on=grouping_col, how='left')

    rep['SCR'] = rep['SCR'].fillna(0)
    final_cols = cols + ['CDI', 'GRS', 'ASR', 'SFI', 'ACE', 'ASI', 'PCD', 'SCR', 'sample']
    return rep[final_cols].sort_values(by='CDI', ascending=False).round(2)

# --- 6. GENERAZIONE ---
full_data = compute_final_metrics(df_merged, ['type', 'agent'], scr_agent_type)
full_data[full_data['sample'] >= ROBUST_THRESHOLD].to_csv(os.path.join(reports_dir, "report_completo_agente_tipo.csv"), index=False)
full_data[full_data['sample'] < ROBUST_THRESHOLD].to_csv(os.path.join(reports_dir, "appendice_bassa_numerosita.csv"), index=False)

compute_final_metrics(df_merged, 'agent', scr_agent).to_csv(os.path.join(reports_dir, "report_aggregato_agenti.csv"), index=False)
compute_final_metrics(df_merged, 'type', scr_type).to_csv(os.path.join(reports_dir, "report_aggregato_tipologia_task.csv"), index=False)

print("\n✅ REPORT GENERATI CON SUCCESSO!")

Calcolo SCR granulare con verifica Overlap Directory...

✅ REPORT GENERATI CON SUCCESSO!
